In [5]:
import re
import warnings
import unicodedata

import numpy as np
import pandas as pd
from sklearn.preprocessing import KBinsDiscretizer
from efficient_apriori import apriori

warnings.filterwarnings('ignore')

In [6]:
BIN_LABELS = ['Low', 'Medium', 'High']

def normalize_name(name):
    if not isinstance(name, str):
        return ''
    name = unicodedata.normalize('NFKD', name).encode('ascii', 'ignore').decode('ascii')
    name = name.strip().upper()
    name = re.sub(r'[^A-Z0-9 ]', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()


def discretize_df(df_in, numeric_cols):
    kbd = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='quantile')
    df_out = df_in.copy()
    disc_cols = []
    for col in numeric_cols:
        if col not in df_out.columns:
            continue
        mask = df_out[col].notna()
        if mask.sum() < 10:
            continue
        col_cat = f'{col}_cat'
        vals = df_out.loc[mask, col].values.reshape(-1, 1)
        bins = kbd.fit_transform(vals).astype(int).flatten()
        df_out.loc[mask, col_cat] = [BIN_LABELS[b] for b in bins]
        disc_cols.append(col_cat)
    return df_out, disc_cols


def wracc(n_dataset, n_sg, n_target_dataset, n_target_sg):
    if n_sg == 0:
        return 0.0
    p0 = n_target_dataset / n_dataset
    p1 = n_target_sg / n_sg
    return (n_sg / n_dataset) * (p1 - p0)


def beam_search(df_in, target_col, feature_cols, depth=3, beam_width=10, top_k=12):
    n = len(df_in)
    n_t = int(df_in[target_col].sum())
    all_selectors = []
    for col in feature_cols:
        for val in df_in[col].dropna().unique():
            all_selectors.append((col, val))

    def apply_selectors(sel_list):
        mask = pd.Series([True] * n, index=df_in.index)
        for col, val in sel_list:
            mask &= df_in[col] == val
        return mask

    def score(sel_list):
        mask = apply_selectors(sel_list)
        n_sg = int(mask.sum())
        n_tsg = int(df_in.loc[mask, target_col].sum())
        return wracc(n, n_sg, n_t, n_tsg), n_sg, n_tsg

    beam = [[]]
    visited = set()
    results = []

    for d in range(1, depth + 1):
        candidates = []
        for current in beam:
            current_cols = {c for c, _ in current}
            for sel in all_selectors:
                if sel[0] in current_cols:
                    continue
                new_sel = sorted(current + [sel], key=lambda x: str(x))
                key = tuple(new_sel)
                if key in visited:
                    continue
                visited.add(key)
                q, n_sg, n_tsg = score(new_sel)
                if n_sg < 25:
                    continue
                candidates.append((q, new_sel, n_sg, n_tsg))
        if not candidates:
            break
        candidates.sort(key=lambda x: -x[0])
        results.extend(candidates)
        beam = [c[1] for c in candidates[:beam_width]]

    results.sort(key=lambda x: -x[0])
    seen = set()
    final = []
    for q, sel, n_sg, n_tsg in results:
        key = tuple(sel)
        if key not in seen:
            seen.add(key)
            mask = apply_selectors(sel)
            cities = df_in.loc[mask, ['nm_municipio', 'sg_uf']].values.tolist()
            final.append((q, sel, n_sg, n_tsg, cities))
        if len(final) == top_k:
            break
    return final


def print_subgroups(results, label, n_total, n_target_total, top=10):
    base_rate = n_target_total / n_total
    print(f'  Base rate ({label}): {base_rate:.1%}  ({n_target_total}/{n_total} municipalities)')
    for rank, (q, sel, n_sg, n_tsg, cities) in enumerate(results[:top], 1):
        desc = ' AND '.join(f"{c.replace('_cat', '')}={v}" for c, v in sel)
        rate = n_tsg / n_sg if n_sg else 0
        city_list = ', '.join(f'{c} ({s})' for c, s in sorted(cities)[:8])
        ellipsis = f' ... +{len(cities) - 8} more' if len(cities) > 8 else ''
        print(f'  #{rank:02d}  WRAcc={q:+.4f}  [{n_tsg}/{n_sg} = {rate:.1%}]')
        print(f'        Pattern : {desc}')
        print(f'        Cities  : {city_list}{ellipsis}')
        print()


def print_rules(rules_list, label, top=10):
    print(f'  --- Association rules -> {label} ---')
    if not rules_list:
        print('  (none found at current thresholds)')
        return
    for rank, (lift, lhs, sup, conf) in enumerate(rules_list[:top], 1):
        lhs_str = ' AND '.join(
            i.replace('_cat=', '=').replace('_cat', '') for i in sorted(lhs)
        )
        print(f'  #{rank:02d}  lift={lift:.2f}  support={sup:.2%}  confidence={conf:.2%}')
        print(f'        IF   {lhs_str}')
        print()


def run_rules(df_in, feat_cols, target_item, min_sup=0.06, min_conf=0.58):
    transactions = []
    for _, row in df_in.iterrows():
        t = []
        for col in feat_cols:
            val = row.get(col)
            if pd.notna(val):
                t.append(f'{col}={val}')
        t.append(target_item)
        transactions.append(tuple(t))
    _, rules_all = apriori(
        transactions, min_support=min_sup, min_confidence=min_conf, max_length=4
    )
    prefix = target_item.split('=')[0] + '='
    r_filtered = []
    for rule in rules_all:
        if set(rule.rhs) != {target_item}:
            continue
        lhs_clean = [
            i for i in rule.lhs
            if not i.startswith(prefix) and not i.startswith('is_')
        ]
        if not lhs_clean:
            continue
        r_filtered.append((rule.lift, lhs_clean, rule.support, rule.confidence))
    r_filtered.sort(key=lambda x: -x[0])
    return r_filtered

In [7]:
df_mun = pd.read_csv('dados_municipios_panorama.csv', encoding='utf-8')
df_votes = pd.read_csv(
    'votacao_candidato_municipio_presidente_2022.csv',
    encoding='latin-1',
    sep=';',
    dtype={'pc_votos_validos': str},
)

df_mun['municipio_norm'] = df_mun['municipio'].apply(normalize_name)
df_votes['municipio_norm'] = df_votes['nm_municipio'].apply(normalize_name)

df_r1 = df_votes[df_votes['nr_turno'] == 1].copy()
df_r1['pc_votos_validos'] = (
    df_r1['pc_votos_validos'].str.replace(',', '.', regex=False).astype(float)
)

pivot = df_r1.pivot_table(
    index=['sg_uf', 'municipio_norm', 'nm_municipio'],
    columns='nm_candidato',
    values='pc_votos_validos',
    aggfunc='sum',
).reset_index()
pivot.columns.name = None
pivot.columns = [
    unicodedata.normalize('NFKD', c).encode('ascii', 'ignore').decode('ascii')
    .upper().replace(' ', '_')
    if c not in ('sg_uf', 'municipio_norm', 'nm_municipio') else c
    for c in pivot.columns
]

BOLSONARO_COL = 'JAIR_MESSIAS_BOLSONARO'
LULA_COL = 'LUIZ_INACIO_LULA_DA_SILVA'


def get_winner(row):
    b = row.get(BOLSONARO_COL, 0) or 0
    l = row.get(LULA_COL, 0) or 0
    if b > l:
        return 'BOLSONARO'
    elif l > b:
        return 'LULA'
    return 'TIE'


pivot['winner'] = pivot.apply(get_winner, axis=1)
df = pivot.merge(df_mun, on='municipio_norm', how='inner')
print(f'Merged rows: {len(df):,}')

Merged rows: 6,251


In [8]:
DROP = [
    'municipio', 'municipio_id', 'municipio_norm',
    'SIMONE_NASSAR_TEBET', 'CIRO_FERREIRA_GOMES', 'SORAYA_VIEIRA_THRONICKE',
    'LUIZ_FELIPE_CHAVES_D_AVILA', 'KELMON_LUIS_DA_SILVA_SOUZA',
    'VERA_LUCIA_PEREIRA_DA_SILVA_SALGADO', 'JOSE_MARIA_EYMAEL',
    'SOFIA_PADUA_MANZANO', 'LEONARDO_PERICLES_VIEIRA_ROQUE',
    'Nome masculino mais popular', 'Nome feminino mais popular',
    'Sobrenome mais popular', 'Sistema Costeiro-Marinho',
    'Hierarquia urbana', 'Região de Influência',
    'Região intermediária', 'Região imediata', 'Mesorregião', 'Microrregião',
    'Total de receitas brutas realizadas', 'Total de despesas brutas empenhadas',
    'Transferências correntes (Percentual em relação às receitas correntes brutas realizadas)',
]
df.drop(columns=[c for c in DROP if c in df.columns], inplace=True, errors='ignore')

RENAME = {
    'População no último censo': 'pop_censo',
    'População estimada': 'pop_estimada',
    'População quilombola': 'pop_quilombola',
    'População indígena': 'pop_indigena',
    'Densidade demográfica': 'dens_demografica',
    'Salário médio mensal dos trabalhadores formais': 'salario_medio',
    'Pessoal ocupado em postos de trabalho formais': 'empregos_formais',
    'Percentual da população com rendimento nominal mensal per capita de até 1/2 salário mínimo': 'pct_baixa_renda',
    'Taxa de escolarização de 6 a 14 anos de idade': 'taxa_escolarizacao',
    'IDEB – Anos iniciais do ensino fundamental (Rede pública)': 'ideb_iniciais',
    'IDEB – Anos finais do ensino fundamental (Rede pública)': 'ideb_finais',
    'Matrículas no ensino fundamental': 'matriculas_fund',
    'Matrículas no ensino médio': 'matriculas_medio',
    'Docentes no ensino fundamental': 'docentes_fund',
    'Docentes no ensino médio': 'docentes_medio',
    'Número de estabelecimentos de ensino fundamental': 'estabs_fund',
    'Número de estabelecimentos de ensino médio': 'estabs_medio',
    'PIB per capita': 'pib_per_capita',
    'Índice de Desenvolvimento Humano Municipal (IDHM)': 'idhm',
    'Mortalidade Infantil': 'mortalidade_infantil',
    'Internações por diarreia pelo SUS': 'internacoes_diarreia',
    'Estabelecimentos de Saúde SUS': 'estabs_sus',
    'Área urbanizada': 'area_urbanizada',
    'Esgotamento sanitário por rede geral, rede pluvial ou fossa ligada à rede': 'saneamento',
    'Arborização de vias públicas': 'arborizacao',
    'Urbanização de vias públicas': 'urbanizacao_vias',
    'População exposta ao risco': 'pop_risco',
    'Área da unidade territorial': 'area_km2',
    'Bioma predominante': 'bioma',
}
df.rename(columns={k: v for k, v in RENAME.items() if k in df.columns}, inplace=True)

NUMERIC_COLS = [
    'pop_censo', 'pop_estimada', 'pop_quilombola', 'pop_indigena',
    'dens_demografica', 'salario_medio', 'empregos_formais',
    'pct_baixa_renda', 'taxa_escolarizacao', 'ideb_iniciais', 'ideb_finais',
    'pib_per_capita', 'idhm', 'mortalidade_infantil', 'internacoes_diarreia',
    'estabs_sus', 'area_urbanizada', 'saneamento', 'arborizacao',
    'urbanizacao_vias', 'pop_risco', 'area_km2',
    BOLSONARO_COL, LULA_COL,
]
NUMERIC_COLS = [c for c in NUMERIC_COLS if c in df.columns]

for col in NUMERIC_COLS:
    df[col] = pd.to_numeric(
        df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce'
    )

df_disc, disc_cols = discretize_df(df, NUMERIC_COLS)
CAT_COLS = [c for c in ['sg_uf', 'bioma'] if c in df_disc.columns]
VOTE_CATS_EXCL = {f'{BOLSONARO_COL}_cat', f'{LULA_COL}_cat'}

In [9]:
COVID_COLS = ['covid_deaths_per_100k', 'covid_cases_per_100k', 'covid_cfr']

covid = pd.read_csv('cases-brazil-cities.csv')
covid['city_only'] = covid['city'].str.split('/').str[0]
covid['municipio_norm'] = covid['city_only'].apply(normalize_name)
covid.rename(columns={
    'deaths_per_100k_inhabitants': 'covid_deaths_per_100k',
    'totalCases_per_100k_inhabitants': 'covid_cases_per_100k',
    'deaths_by_totalCases': 'covid_cfr',
}, inplace=True)
covid_slim = covid[['municipio_norm'] + COVID_COLS].copy()
print(f'COVID rows loaded: {len(covid_slim):,}')

df_disc2 = df_disc.copy()
df_disc2['municipio_norm'] = df_disc2['nm_municipio'].apply(normalize_name)
df_disc2 = df_disc2.merge(covid_slim, on='municipio_norm', how='left')

df_disc2_d, disc_cols_cov2 = discretize_df(df_disc2, COVID_COLS)
print(f'Municipalities with COVID data: {df_disc2_d["covid_deaths_per_100k_cat"].notna().sum()}')

COVID rows loaded: 5,596
Municipalities with COVID data: 8042


In [10]:
all_disc2_cols = list(dict.fromkeys(
    [c for c in disc_cols + disc_cols_cov2 if c not in VOTE_CATS_EXCL] + CAT_COLS
))
_keep2 = list(dict.fromkeys(all_disc2_cols + ['nm_municipio', 'sg_uf', 'winner']))
df_analysis2 = df_disc2_d[[c for c in _keep2 if c in df_disc2_d.columns]].copy()
df_analysis2 = df_analysis2.loc[:, ~df_analysis2.columns.duplicated()]
df_analysis2.reset_index(drop=True, inplace=True)
feat_cols2 = [c for c in all_disc2_cols if c in df_analysis2.columns]

df_analysis2['bolsonaro_won'] = (df_analysis2['winner'] == 'BOLSONARO').astype(int)
df_analysis2['lula_won'] = (df_analysis2['winner'] == 'LULA').astype(int)

print('=' * 70)
print('PART A: 2022 ELECTION -- BOLSONARO vs LULA (with COVID features)')
print('=' * 70)

print('\n--- Subgroup discovery: BOLSONARO wins ---')
sg_b2 = beam_search(df_analysis2, 'bolsonaro_won', feat_cols2)
print_subgroups(sg_b2, 'Bolsonaro', len(df_analysis2), int(df_analysis2['bolsonaro_won'].sum()))

transactions_2022 = []
for _, row in df_analysis2.iterrows():
    t = [f'{col}={row[col]}' for col in feat_cols2 if pd.notna(row.get(col))]
    t.append(f'winner={row["winner"]}')
    transactions_2022.append(tuple(t))
_, rules_all_2022 = apriori(
    transactions_2022, min_support=0.06, min_confidence=0.60, max_length=4
)


def filter_winner_rules(rules_all, winner_val):
    target = f'winner={winner_val}'
    out = []
    for rule in rules_all:
        if set(rule.rhs) != {target}:
            continue
        lhs_clean = [
            i for i in rule.lhs
            if not i.startswith('winner=') and not i.startswith('is_')
        ]
        if not lhs_clean:
            continue
        out.append((rule.lift, lhs_clean, rule.support, rule.confidence))
    out.sort(key=lambda x: -x[0])
    return out


print('\n--- Association rules: BOLSONARO wins ---')
print_rules(filter_winner_rules(rules_all_2022, 'BOLSONARO'), 'BOLSONARO')

print('\n--- Subgroup discovery: LULA wins ---')
sg_l2 = beam_search(df_analysis2, 'lula_won', feat_cols2)
print_subgroups(sg_l2, 'Lula', len(df_analysis2), int(df_analysis2['lula_won'].sum()))

print('\n--- Association rules: LULA wins ---')
print_rules(filter_winner_rules(rules_all_2022, 'LULA'), 'LULA')

PART A: 2022 ELECTION -- BOLSONARO vs LULA (with COVID features)

--- Subgroup discovery: BOLSONARO wins ---
  Base rate (Bolsonaro): 37.5%  (3017/8042 municipalities)
  #01  WRAcc=+0.1035  [1831/2662 = 68.8%]
        Pattern : pct_baixa_renda=Low
        Cities  : ABADIA DE GOIÁS (GO), ABAETÉ (MG), ADAMANTINA (SP), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUANIL (MG), AGUAÍ (SP) ... +2654 more

  #02  WRAcc=+0.1030  [1827/2661 = 68.7%]
        Pattern : idhm=High
        Cities  : ABADIA DE GOIÁS (GO), ADAMANTINA (SP), ADELÂNDIA (GO), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUAÍ (SP), AGUDOS (SP) ... +2653 more

  #03  WRAcc=+0.0950  [1577/2167 = 72.8%]
        Pattern : idhm=High AND pct_baixa_renda=Low
        Cities  : ABADIA DE GOIÁS (GO), ADAMANTINA (SP), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUAÍ (SP), AGUDOS (SP), AJURICABA (RS) ... +2159 more

  #04  WRAcc=+0.0930  [1592/2250 = 70.8%]
        Pattern : pct_baixa_renda=Low AND pop_quilombola=nan
       

In [11]:
df18 = pd.read_csv(
    'votacao_candidato_municipio_presidente_2018.csv',
    encoding='latin-1',
    sep=';',
)
df18.columns = df18.columns.str.lower()
df18['municipio_norm'] = df18['nm_municipio'].apply(normalize_name)

df18_r1 = df18[df18['nr_turno'] == 1].copy()
pivot18 = df18_r1.pivot_table(
    index=['sg_uf', 'municipio_norm', 'nm_municipio'],
    columns='nm_candidato',
    values='qt_votos_nominais_validos',
    aggfunc='sum',
).reset_index()
pivot18.columns.name = None

BOLS18 = 'JAIR MESSIAS BOLSONARO'


def winner_2018(row):
    b = row.get(BOLS18, 0) or 0
    h = row.get('FERNANDO HADDAD', 0) or 0
    return 'BOLSONARO' if b >= h else 'HADDAD'


pivot18['winner_2018'] = pivot18.apply(winner_2018, axis=1)
print(
    f"2018: {len(pivot18):,} municipalities  "
    f"(Bolsonaro: {(pivot18.winner_2018 == 'BOLSONARO').sum()}, "
    f"Haddad: {(pivot18.winner_2018 == 'HADDAD').sum()})"
)

df22_key = df_analysis2[['nm_municipio', 'winner']].copy()
df22_key.rename(columns={'winner': 'winner_2022'}, inplace=True)
df22_key['municipio_norm'] = df22_key['nm_municipio'].apply(normalize_name)

merge_key = pivot18[['municipio_norm', 'winner_2018']].copy()
df22_key2 = df22_key[['municipio_norm', 'winner_2022']].drop_duplicates('municipio_norm')
cross = df22_key2.merge(merge_key, on='municipio_norm', how='inner')
cross['trajectory'] = cross['winner_2018'] + '_' + cross['winner_2022']

print('Trajectory counts:')
for t, c in cross['trajectory'].value_counts().items():
    print(f'  {t:<30} {c:>4}')

2018: 5,708 municipalities  (Bolsonaro: 2987, Haddad: 2721)
Trajectory counts:
  HADDAD_LULA                    2661
  BOLSONARO_BOLSONARO            2078
  BOLSONARO_LULA                  781
  HADDAD_BOLSONARO                 41
  BOLSONARO_TIE                     3


In [12]:
NUMERIC_TRAJ = [c for c in [
    'pop_censo', 'pop_estimada', 'dens_demografica', 'salario_medio',
    'empregos_formais', 'pct_baixa_renda', 'taxa_escolarizacao',
    'ideb_iniciais', 'ideb_finais', 'pib_per_capita', 'idhm',
    'mortalidade_infantil', 'estabs_sus', 'saneamento', 'arborizacao',
    'urbanizacao_vias', 'area_km2',
] if c in df_disc.columns]

df_base = df_disc.copy()
df_base['municipio_norm'] = df_base['nm_municipio'].apply(normalize_name)
df_base = df_base.merge(covid_slim, on='municipio_norm', how='left')

for col in NUMERIC_TRAJ + COVID_COLS:
    if col in df_base.columns:
        df_base[col] = pd.to_numeric(
            df_base[col].astype(str).str.replace(',', '.', regex=False), errors='coerce'
        )

df_traj_base, disc_traj = discretize_df(df_base, NUMERIC_TRAJ + COVID_COLS)
df_traj_base['municipio_norm'] = df_traj_base['nm_municipio'].apply(normalize_name)
df_traj = df_traj_base.merge(
    cross[['municipio_norm', 'trajectory', 'winner_2018', 'winner_2022']],
    on='municipio_norm',
    how='inner',
)
df_traj.reset_index(drop=True, inplace=True)
print(f'Trajectory+COVID frame: {len(df_traj):,} municipalities')

CAT_COLS2 = [c for c in ['sg_uf', 'bioma'] if c in df_traj.columns]
FEAT_TRAJ = list(dict.fromkeys(disc_traj + CAT_COLS2))

TRAJECTORIES = [
    ('BOLSONARO_BOLSONARO', 'Bolsonaro->Bolsonaro (stayed right)'),
    ('HADDAD_LULA', 'Haddad->Lula (stayed left)'),
    ('BOLSONARO_LULA', 'Bolsonaro->Lula (swung left)'),
    ('HADDAD_BOLSONARO', 'Haddad->Bolsonaro (swung right)'),
]

sg_traj = {}
rules_traj = {}

print('\n' + '=' * 70)
print('PART B: 2018->2022 TRAJECTORIES (with COVID features)')
print('=' * 70)

for traj_code, traj_label in TRAJECTORIES:
    n_traj = (df_traj['trajectory'] == traj_code).sum()
    if n_traj < 20:
        print(f'  Skipping {traj_label} -- only {n_traj} municipalities.')
        continue
    print(f'\n{"=" * 70}')
    print(f'TRAJECTORY (w/ COVID): {traj_label}')
    print('=' * 70)
    target_col = f'is_{traj_code}'
    df_traj[target_col] = (df_traj['trajectory'] == traj_code).astype(int)
    sg = beam_search(df_traj, target_col, FEAT_TRAJ)
    sg_traj[traj_code] = sg
    print_subgroups(sg, traj_label, len(df_traj), int(df_traj[target_col].sum()))
    traj_item = f'trajectory={traj_code}'
    r_filtered = run_rules(df_traj, FEAT_TRAJ, traj_item, min_sup=0.04, min_conf=0.55)
    rules_traj[traj_code] = r_filtered
    print_rules(r_filtered, traj_label)

Trajectory+COVID frame: 13,653 municipalities

PART B: 2018->2022 TRAJECTORIES (with COVID features)

TRAJECTORY (w/ COVID): Bolsonaro->Bolsonaro (stayed right)
  Base rate (Bolsonaro->Bolsonaro (stayed right)): 21.3%  (2908/13653 municipalities)
  #01  WRAcc=+0.0750  [1946/4330 = 44.9%]
        Pattern : idhm=High
        Cities  : ABADIA DE GOIÁS (GO), ADAMANTINA (SP), ADELÂNDIA (GO), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUAÍ (SP), AGUDOS (SP) ... +4322 more

  #02  WRAcc=+0.0711  [1916/4438 = 43.2%]
        Pattern : pct_baixa_renda=Low
        Cities  : ABADIA DE GOIÁS (GO), ABAETÉ (MG), ADAMANTINA (SP), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUANIL (MG), AGUAÍ (SP) ... +4430 more

  #03  WRAcc=+0.0652  [1672/3671 = 45.5%]
        Pattern : idhm=High AND pct_baixa_renda=Low
        Cities  : ABADIA DE GOIÁS (GO), ADAMANTINA (SP), ADOLFO (SP), AGROLÂNDIA (SC), AGRONÔMICA (SC), AGUAÍ (SP), AGUDOS (SP), AJURICABA (RS) ... +3663 more

  #04  WRAcc=+0.0582  [1677/414

In [13]:
print('=' * 70)
print('PART C: CONTRASTIVE SUMMARY -- COVID features across trajectories')
print('=' * 70)


def top_patterns(sg_list, top=5):
    out = []
    for q, sel, n_sg, n_tsg, _ in sg_list[:top]:
        desc = ' & '.join(
            f"{c.replace('_cat', '').replace('_', ' ')}={v}" for c, v in sel
        )
        out.append(f'  {desc}  [{n_tsg}/{n_sg} = {n_tsg / n_sg:.0%}]')
    return '\n'.join(out) if out else '  (none)'


for traj_code, traj_label in TRAJECTORIES:
    if traj_code not in sg_traj:
        continue
    print(f'\n-- {traj_label} --')
    print(top_patterns(sg_traj[traj_code]))

for traj_code, traj_label in TRAJECTORIES:
    if traj_code not in sg_traj:
        continue
    sg = sg_traj[traj_code]
    covid_patterns = [
        (q, sel, n_sg, n_tsg, cities)
        for q, sel, n_sg, n_tsg, cities in sg
        if any('covid' in c for c, _ in sel)
    ]
    if covid_patterns:
        print(f'\n  COVID patterns in {traj_label}:')
        print(top_patterns(covid_patterns, top=3))

PART C: CONTRASTIVE SUMMARY -- COVID features across trajectories

-- Bolsonaro->Bolsonaro (stayed right) --
  idhm=High  [1946/4330 = 45%]
  pct baixa renda=Low  [1916/4438 = 43%]
  idhm=High & pct baixa renda=Low  [1672/3671 = 46%]
  covid cases per 100k=High  [1677/4146 = 40%]
  bioma=Mata Atlântica & idhm=High  [1508/3376 = 45%]

-- Haddad->Lula (stayed left) --
  pct baixa renda=High  [3915/5000 = 78%]
  idhm=Low  [3871/5055 = 77%]
  idhm=Low & pct baixa renda=High  [3349/4312 = 78%]
  bioma=Caatinga  [2803/3613 = 78%]
  ideb iniciais=Low  [3405/4768 = 71%]

-- Bolsonaro->Lula (swung left) --
  saneamento=High  [1222/4330 = 28%]
  bioma=Mata Atlântica  [1684/6325 = 27%]
  pct baixa renda=Low  [1216/4438 = 27%]
  dens demografica=Medium  [1239/4578 = 27%]
  bioma=Mata Atlântica & dens demografica=Medium  [743/2502 = 30%]

-- Haddad->Bolsonaro (swung right) --
  saneamento=Low  [104/4535 = 2%]
  pop censo=Low & pop estimada=Low & salario medio=High  [62/1762 = 4%]
  pop censo=Low & 